<a href="https://colab.research.google.com/github/deckinhow/repositorio_grupo4/blob/develop/notebooks/02_Derick.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook Derick

## 0. Setup Inicial

Organizando o notebook para trabalho.

Etapas:
- Linkando o notebook com o repositório do GitHub;
- Importando as bibliotecas que serão utilizadas

In [1]:
%%capture

import os

#Clonar o repositório do GitHub
if not os.path.exists('repositorio_grupo4/'):
  !git clone {'https://github.com/deckinhow/repositorio_grupo4.git'}
  %cd {'repositorio_grupo4/'}
else:
  %cd {'repositorio_grupo4/'}
  !git pull

#Bibliotecas que serão utilizadas
from dados_e_funcoes import funcoes #Funções criadas por nós
import statistics
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import importlib
import yfinance
import cvxpy as cp
!pip install gurobipy
import gurobipy

#Atualizar as funções salvas no arquivo de funções
importlib.reload(funcoes)

#Remover mensagens de aviso
warnings.filterwarnings('ignore')

## 1. Setup do Banco de Dados

Importando a database.

Etapas:
- Importar os dados

## 2. Pré processamento


### 2.1 Análise incial dos dados

Etapas:
- Observar os dados
- Procurar anomalias e dados "NA"


### 2.2 Limpeza e tratamento

Etapas:
- Limpeza: remover dados discrepantes, outliers e quaiquer outros dados não relevantes;
- Tratamento de dados "NA"

## 3 Análise Descritiva e Exploratória

## Modelo matemático de Index Tracking


**Formulação Matemática**
$$\min \frac{1}{T} \sum_{t=1}^{T} (\sum_{i \in I} w_i r_{t,i} - R_t)^2$$

Onde

$I$: Conjunto de ativos disponíveis

$T$: Número de períodos

$w_i$: Peso do ativo $i$ no portfólio de tracking

$z_i$: Varável binária para o ativo $i$

$R_t$: Rendimento do índice no período $t$

$r_t$: Rendimento do ativo $i$ no período $t$

$K$: O número máximo de ativos permitidos


**Restrições matemáticas:**
1. $\sum_{i \in I} w_i = 1$
2. $w_i \ge 0$
3. $w_i \le z_i$ e $\sum_{i \in I} z_i \le K$, onde $z_i \in \{0,1\}$


Veja que, a função matemática visa minimizar o Erro Quadrático Médio.

In [2]:

def otimizador(retornos_ativos, retornos_indice, K):
    """
    Otimiza o portfólio rastreador limitando o número máximo de ativos.

    Seja retornos_ativos o data frame dos retornos dos ativos.
    Seja retornos_indice o data frame dos retornos do índice de referência
    (Ex.: IBOVESPA e S&P100)
    Seja K o número máximo de ativos permitidos.
    """

    n_ativos = retornos_ativos.shape[1] #Número de ativos


    # 1. Variáveis de decisão


    w = cp.Variable(n_ativos) # Vetor de pesos (variáveis contínuas)
    y = cp.Variable(n_ativos, boolean=True) # Vetor de escolhas (0 ou 1)

    # 2. Criação do modelo matemático

    ret_portfolio = retornos_ativos.values @ w
    diferenca = ret_portfolio - retornos_indice.values.flatten()

    # Função minimizadora do Erro Quadrático Médio
    T = retornos_ativos.shape[0]
    min_eqm = cp.Minimize(cp.sum_squares(diferenca) / T)


    # 3. Restrições matemáticas do modelo


    #Sem as restrições, teremos problemas com o modelo
    restricoes = [
        cp.sum(w) == 1,   # Restrição 1: Orçamento total de 100%
        w >= 0,           # Restrição 2: Apenas posições compradas
        w <= y,           # Restrição 3: O peso w_i só pode ser maior que 0 se y_i for 1.
        cp.sum(y) <= K    # Restrição 4: A soma das ações escolhidas não pode passar de K.
    ]


    # 4. O Modelo Final

    modelo = cp.Problem(min_eqm, restricoes)

    # Como adicionamos uma variável booleana, o cvxpy vai procurar um solver
    # capaz de lidar com inteiros (MIQP), como o SCIP, GLPK_MI ou o próprio Gurobi.

    print(f"Resolvendo modelo para no máximo {K} ativos...")
    modelo.solve(solver='GUROBI') #Põe o modelo em prática


    # 5. Limpeza

    # Zera resquícios matemáticos (pesos menores que 0.01%).
    pesos_limpos = np.where(w.value < 1e-4, 0, w.value)

    # Re-normaliza para garantir que a soma é exatamente 1.
    pesos_finais = pesos_limpos / np.sum(pesos_limpos)

    return pesos_finais


In [6]:
retornos_ativos = pd.read_csv('https://raw.githubusercontent.com/deckinhow/repositorio_grupo4/develop/dados_e_funcoes/ibov_precos.csv', index_col='Date', parse_dates=True)
retornos_indice = pd.read_csv('https://raw.githubusercontent.com/deckinhow/repositorio_grupo4/develop/dados_e_funcoes/ibov_indice.csv', index_col='Date', parse_dates=True)

qtd_nan_ativos = retornos_ativos.isna().sum().sum()
qtd_inf_ativos = np.isinf(retornos_ativos).sum().sum()

print(retornos_indice.shape)

print(f"Total de NaNs nas ações: {qtd_nan_ativos}")
print(f"Total de Infs nas ações: {qtd_inf_ativos}")
print(f"Formato final limpo: {retornos_ativos.shape}")




(1737, 1)
Total de NaNs nas ações: 206440
Total de Infs nas ações: 0
Formato final limpo: (1738, 490)


In [4]:
# Configuração do Backtest
dias_por_ano = 252
tamanho_treino = dias_por_ano * 2
tamanho_teste = dias_por_ano * 1
numero_de_janelas = 5
K_maximo = 15 # O Dev 3 pode testar diferentes Ks aqui

resultados_metricas = []
retornos_out_of_sample = pd.Series(dtype=float)

for i in range(numero_de_janelas):
    inicio_treino = i * tamanho_teste
    fim_treino = inicio_treino + tamanho_treino
    fim_teste = fim_treino + tamanho_teste

    # Separando Treino e Teste
    ativos_treino = retornos_ativos.iloc[inicio_treino:fim_treino]
    bench_treino = retornos_indice.iloc[inicio_treino:fim_treino]
    ativos_teste = retornos_ativos.iloc[fim_treino:fim_teste]
    bench_teste = retornos_indice.iloc[fim_treino:fim_teste]

    # Otimizando Pesos
    pesos_ideais = otimizador(ativos_treino, bench_treino, K_maximo)

    # Calculando Performance Relativa
    retornos_carteira_teste = ativos_teste @ pesos_ideais
    retornos_out_of_sample = pd.concat([retornos_out_of_sample, retornos_carteira_teste])

    # Calculando Métricas (Tracking Error Anualizado)
    erro_diario = retornos_carteira_teste - bench_teste.values.flatten()
    te_anualizado = np.std(erro_diario) * np.sqrt(dias_por_ano)

    resultados_metricas.append({
        'Janela de Teste': i + 1,
        'Tracking Error (%)': round(te_anualizado * 100, 2)
    })

# Exibindo as métricas para o Visual 1 interpretar depois
df_metricas = pd.DataFrame(resultados_metricas)
display(df_metricas)

Resolvendo modelo para no máximo 15 ativos...
Restricted license - for non-production use only - expires 2027-11-29


GurobiError: Element 1 of a double array is Nan or Inf.